# Tracer + ToyAgent — audit your own toy agent

This notebook walks you through **defining a toy agent as a sequence of Python steps** and **auditing it with Tracer's LLM judge**. By the end you will have:

1. a tiny agent you wrote,
2. a step-level blame report from Tracer you can drop straight into your Week 5 slides.

The pattern is **backbone-agnostic**: your step functions can call OpenAI, Anthropic, a local model, or nothing at all. Tracer only sees each step's inputs and outputs.

## 0. Setup

Make sure:
- `pandas` and `openai` are installed (`pip install -r requirements.txt`),
- you are running this notebook from inside the Tracer repo at `examples/tracer_toy_agent/` **or** can reach the network (the next cell auto-clones Tracer into a local `_tracer/` folder if the parent layout isn't recognized — handy for Colab),
- your OpenAI key is available **either** in `./api_key.json` as `{"api_key": "sk-..."}` **or** in the `OPENAI_API_KEY` environment variable.

> `api_key.json` is git-ignored — keep your key in the file, never paste it into a cell.

In [ ]:
import sys                                                                                                                                                                    
!{sys.executable} -m pip install "openai>=1.0.0" "pandas>=1.5.0"

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# This notebook lives at Tracer/examples/tracer_toy_agent/ in the Tracer
# repo, so the Tracer modules (parser.py, executor.py, ...) sit at
# parent.parent. When the notebook is opened standalone (e.g. in a fresh
# Colab session), that path won't exist -- we clone Tracer as a fallback.
CANDIDATE = Path.cwd().resolve().parent.parent
if (CANDIDATE / "parser.py").exists() and (CANDIDATE / "executor.py").exists():
    TRACER_DIR = CANDIDATE
else:
    TRACER_DIR = Path.cwd().resolve() / "_tracer"
    if not TRACER_DIR.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/Yungxi/Tracer.git", str(TRACER_DIR)],
            check=True,
        )

assert (TRACER_DIR / "parser.py").exists(), f"Tracer not found at {TRACER_DIR}"
if str(TRACER_DIR) not in sys.path:
    sys.path.insert(0, str(TRACER_DIR))

# Make our ToyAgent helper importable (this notebook's directory).
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from toy_agent import ToyAgent             # our ~120-line helper
from parser import parse_source             # Tracer modules
from executor import TracingExecutor
from judge import LLMJudge
from reporter import Reporter

print("Tracer dir:", TRACER_DIR)
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

## 1. Define your agent's goal

The **goal** is the one-sentence description Tracer's judge will evaluate each step against. Be specific about what `CORRECT` means.

In [ ]:
agent = ToyAgent(
    goal="Find the top 3 customers by total spending and return their names, highest spender first.",
)

## 2. Register steps

Two pieces of setup first:

- **`agent.imports`** — import lines prepended to the generated script.
- **`agent.setup`** — a block of Python code that creates a variable named `state`. This is whatever the *first* step will receive as input. Any tables, constants, or LLM clients your pipeline needs should be reachable from `state`.

**Convention:** each step takes the previous step's return value and returns whatever the next step will consume. By convention `state` is a dict the steps mutate, but any threaded value works. The last step can return the final answer directly.

Why the `state` dict instead of closures over module-level tables? Tracer wraps each step function at *definition time* — once wrapped, the function can no longer see variables declared in the enclosing module. Threading `state` through the arguments sidesteps that. Give each step a docstring that says what it reads from `state` and what it writes back — the judge uses it as context.

In [ ]:
agent.imports = [
    "import pandas as pd",
]

# `setup` must create a variable named `state` --- the initial input for
# the first step. We bundle both tables into it.
agent.setup = """
    state = {
        'customers': pd.DataFrame({
            'customer_id': [1, 2, 3, 4, 5],
            'name':        ['Alice', 'Bob', 'Carol', 'David', 'Eve'],
        }),
        'orders': pd.DataFrame({
            'order_id':     [101, 102, 103, 104, 105, 106],
            'customer_id':  [  1,   2,   1,   3,   2,   4],
            'total_amount': [150.0, 200.0, 300.0, 175.5, 250.0, 400.0],
        }),
    }
"""

In [ ]:
@agent.add_step
def build_totals(state):
    """Aggregate total spending per customer.

    Adds state['totals']: a DataFrame with ['customer_id', 'total_amount'].
    """
    state['totals'] = (
        state['orders']
        .groupby('customer_id')['total_amount']
        .sum()
        .reset_index()
    )
    return state


@agent.add_step
def pick_top_n(state):
    """Pick the top 3 customers by total spending.

    Reads state['totals']; adds state['top'] sorted HIGHEST to LOWEST.
    """
    # BUG (intentional): ascending=True returns the three LOWEST spenders.
    state['top'] = state['totals'].sort_values('total_amount', ascending=True).head(3)
    return state


@agent.add_step
def format_result(state):
    """Join customer names and return the top-3 names as a list (final output)."""
    merged = state['customers'].merge(state['top'], on='customer_id')
    return merged['name'].tolist()


print("Registered steps:", [s.name for s in agent.steps])

## 3. Sanity-run the agent (no Tracer yet)

Before auditing, just run the agent end-to-end. The point is to confirm it **finishes without crashing** — that is exactly the kind of silent failure error localization exists for.

In [ ]:
print("agent.run_local() =>", agent.run_local())
# Expected (with the intentional bug): ['Alice', 'Carol', 'David'] --- the LOWEST spenders.

## 4. See what Tracer will see

`agent.to_script()` stitches imports + init_code + step sources + a small runner into one Python source string. That string is what Tracer's parser and executor will consume.

In [ ]:
print(agent.to_script())

## 5. Audit with Tracer

One call: parse the script, execute step-by-step with the judge attached, report findings. `continue_on_error=True` makes Tracer keep going past errors so we collect **every** bug in one pass.

In [ ]:
import json
from pathlib import Path

# Load the key from api_key.json if present; otherwise fall back to the env var.
# api_key.json is git-ignored so the key never leaves your machine.
key_file = Path.cwd() / "api_key.json"
if key_file.exists():
    api_key = json.loads(key_file.read_text())["api_key"]
else:
    api_key = os.environ.get("OPENAI_API_KEY")

if not api_key:
    raise RuntimeError(
        "No OpenAI key found. Create api_key.json with {\"api_key\": \"sk-...\"} "
        "in this directory, or set OPENAI_API_KEY in your environment."
    )

result = agent.audit_with_tracer(api_key=api_key, model="gpt-5.4-mini-2026-03-17")

## 6. Read the findings

Each entry in `result.errors` has a line number, error type, and message. The logical errors are the interesting ones — they are the silent bugs that `run_local` did not surface.

In [ ]:
print(f"Tracer found {len(result.errors)} issue(s):\n")
for err in result.errors:
    print(f"  line {err.lineno:>3}  {err.error_type:<12}  {err.error_message}")

## 7. Upgrade: property-based localization

Vanilla `LLMJudge` evaluates each step's I/O against the natural-language goal — flexible but noisy. Tracer ships two stronger variants:

- **`PropertyAwareJudge`** — derive 3-6 invariants once (via `PropertyGenerator`), keep them in every judge prompt as context.
- **`TestPropertyJudge`** — on top of that, evaluate each invariant's one-line Python assertion against the observed call; pass/fail flags are fed to the judge as concrete evidence.

Executable FAILs become ground truth; the LLM narrates rather than decides alone. Below we re-audit the same buggy agent with the strongest variant.

In [ ]:
from parser import parse_source
from executor import TracingExecutor
from properties import PropertyGenerator
from judge_test_property import TestPropertyJudge

script = agent.to_script()

# 1. Derive properties once from the goal + script source.
gen = PropertyGenerator(api_key=api_key, model="gpt-5.4-mini-2026-03-17")
props = gen.generate(agent.goal, script)
print(f"Derived {len(props)} properties:")
for p in props:
    tgt = ", ".join(p.applies_to) if p.applies_to else "any step"
    print(f"  {p.id} [{tgt}]: {p.description}")
    if p.assertion:
        print(f"        check: {p.assertion}")

# 2. Audit with TestPropertyJudge: properties + per-call assertion outcomes.
judge = TestPropertyJudge(api_key=api_key, properties=props,
                         model="gpt-5.4-mini-2026-03-17", script_goal=agent.goal)
parsed = parse_source(script)
prop_result = TracingExecutor(parsed, judge=judge, continue_on_error=True).execute()

print(f"\nTestPropertyJudge found {len(prop_result.errors)} issue(s):")
for err in prop_result.errors:
    print(f"  line {err.lineno:>3}  {err.error_message[:200]}")

print("\nPer-call assertion outcomes:")
for name, outcomes in judge.history:
    passes = [o.property_id for o in outcomes if o.status == "pass"]
    fails  = [o.property_id for o in outcomes if o.status == "fail"]
    print(f"  {name}: pass={passes} fail={fails}")

## Next steps — for your Week 5 demo

1. **Swap in your own agent**: replace the three example steps with whatever your team's agent actually does. Keep each step small and give it a tight docstring.
2. **Keep the goal specific**: "return top 3 customers, highest first" is auditable; "be helpful" is not.
3. **Pick a judge**:
   - `LLMJudge` for a quick first pass,
   - `PropertyAwareJudge` if you want the judge to stay anchored to explicit invariants,
   - `TestPropertyJudge` for the strongest evidence — each step annotated with which assertions pass/fail.
4. **Screenshot the `result.errors` list + per-call assertion outcomes** — that is your Week 5 slide.
5. If you want more than logical-error detection, point Tracer at your agent's emitted scripts directly via the CLI: `python tracer.py mine.py --goal "…"` (see the top-level [Tracer README](../../README.md)).

That's it. The whole auditing layer is roughly 20 lines of glue plus Tracer.